In [18]:
import pandas as pd
import altair as alt

# replace with most recent csv
df = pd.read_csv('../data/ActivitiesMar25.csv')

# filter to only running
df = df[df['Activity Type'] == 'Running']

# format location text
df['Location'] = df['Title'].str.replace('Running', '').str.strip()

# date/time conversions
df['Date'] = pd.to_datetime(df['Date'])

def time_to_minutes(pace):
    """Convert Avg Pace (in M:SS) to a numeric value (total minutes)"""
    parts = pace.split(':')
    if len(parts) == 2:
        minutes = float(parts[0])
        seconds = float(parts[1])
        return minutes + seconds/60
    return None

def time_to_hours(pace):
    """Convert Avg Pace (in HH:MM:SS) to a numeric value (total hours)"""
    parts = pace.split(':')
    if len(parts) == 3:
        hours = float(parts[0])
        minutes = float(parts[1])
        seconds = float(parts[2])
        return hours + minutes/60 + seconds/3600
    return None

df['avg_pace_minutes'] = df['Avg Pace'].apply(time_to_minutes)
df['time_hours'] = df['Time'].apply(time_to_hours)

df['cal/hr'] = df['Calories'] / df['time_hours']


brush = alt.selection_interval(
    encodings=['x','y'],
    resolve='intersect')

col = alt.condition(brush, alt.Color('Location:N'), alt.value('gray'))

# Base histogram: shows all data in gray
p11_base = alt.Chart(df).mark_bar(color='gray').encode(
    x=alt.X('Distance:Q', bin=alt.Bin(maxbins=30), title='Distance'),
    y=alt.Y('count()', title='Count')
)

# Overlay: shows only the selected data, colored by location
p11_selected = alt.Chart(df).mark_bar().encode(
    x=alt.X('Distance:Q', bin=alt.Bin(maxbins=30)),
    y='count()',
    color=alt.Color('Location:N')
).transform_filter(brush)

# Combine the layers
p11 = (p11_base + p11_selected).add_params(brush).properties(width=350, height=260)

p21 = alt.Chart(df).mark_circle().encode(
    x='Distance:Q',y='Avg HR:Q',color=col
).add_params(brush).properties(width=250,height=160)

p12 = alt.Chart(df).mark_circle().encode(
    x='Distance:Q',y='Calories:Q',color=col
).add_params(brush).properties(width=180,height=270)

p22 = alt.Chart(df).mark_circle().encode(
    x='Distance:Q',y='Calories:Q',color=col
).add_params(brush).properties(width=250,height=160)

p31 = alt.Chart(df).mark_circle().encode(
    x='Distance:Q',y='avg_pace_minutes:Q',color=col
).add_params(brush).properties(width=250,height=160)

p32 = alt.Chart(df).mark_circle().encode(
    x='Distance:Q',y='cal/hr:Q',color=col
).add_params(brush).properties(width=250,height=160)

r1 = alt.hconcat(p11,p12)
r2 = alt.hconcat(p21,p22)
r3 = alt.hconcat(p31,p32)
fig = alt.vconcat(r1,r2,r3)

fig

TypeError: unsupported operand type(s) for /: 'str' and 'float'